## Demo notebook: Cell Segmentation w/ CyCIF image patches

- Packages: [Cellpose](https://github.com/MouseLand/cellpose); tifffile; [PyTorch](https://pytorch.org/get-started/locally/) (optional: [gcsfs](https://gcsfs.readthedocs.io/en/latest/) if loading data from gcloud)
- Channels: B-catenin (**Cyto segmentation**); DAPI (**nuclei segmentation**)

### Load packages

In [ ]:
import os
import sys
import tifffile
# import gcsfs

import numpy as np
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET

import torch
torch.manual_seed(42)

In [ ]:
from ipywidgets import interact, widgets
from IPython.display import display

from matplotlib import rcParams
rcParams.update({'font.size': 10})
rcParams.update({'figure.dpi': 300})
rcParams.update({'savefig.dpi': 300})

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Import Cellpose modules
from cellpose import utils as cp_utils
from cellpose import plot as cp_plot
from cellpose.models import CellposeModel


### Directory configs

### Configuration Settings

- Model (default / pretrained) & dataset paths
- Model parameter configs

In [ ]:
desc_width = '250px'
layout_width = '500px'

# Path
base_model = widgets.Dropdown(
    options=['nuclei', 'cyto2', 'tissuenet', 'livecell'],
    value='cyto2',
    description="Base Model:",
    layout = widgets.Layout(width=layout_width)
)
base_model.style.description_width = desc_width

pretrained_model_path = widgets.Text(
    value='',
    description="(Optional) Pretrained Model path:",
    layout = widgets.Layout(width=layout_width)
)
pretrained_model_path.style.description_width = desc_width

image_path = widgets.Text(
    value='',
    description="Image path:",
    layout = widgets.Layout(width=layout_width)
)
image_path.style.description_width = desc_width

# Params
diameter = widgets.IntSlider(
    value=30, min=20, max=200, step=2, 
    description="Avg. Cell Diameter:",
    layout = widgets.Layout(width=layout_width)
)
diameter.style.description_width = desc_width

rescale = widgets.FloatSlider(
    value=1.0, min=0.2, max=4.0, step=0.1,
    description="Image rescaling factor:",
    layout = widgets.Layout(width=layout_width)
)
rescale.style.description_width = desc_width

flow_threshold = widgets.FloatSlider(
    value=1.0, min=0.5, max=10.0, step=0.1,
    description="Flow Threshold:",
    layout = widgets.Layout(width=layout_width)
)
flow_threshold.style.description_width = desc_width

min_size = widgets.IntSlider(
    value=200, min=50, max=1000, step=50,
    description="Min Size:",
    layout = widgets.Layout(width=layout_width)
)
min_size.style.description_width = desc_width

channels = widgets.Dropdown(
    options=['2 3', '0 0'],
    value='2 3',
    description="Channels (RGB/Grayscale):",
    layout = widgets.Layout(width=layout_width)
)
channels.style.description_width = desc_width


def check_path_exists(name, path):
    """
    Function to check if a given path exists.
    """
    if not os.path.exists(path):
        print(f"{name} '{path}' does not exist.")
    else:
        print()

@interact
def config_variables(
    base_model=base_model,
    pretrained_model_path=pretrained_model_path,
    image_path=image_path,
    diameter=diameter,
    rescale=rescale,
    flow_threshold=flow_threshold,
    min_size=min_size,
    channels=channels
):
    if len(pretrained_model_path) > 0:
        check_path_exists('Pretrained_model', pretrained_model_path)
    if len(image_path) > 0:
        check_path_exists('Image', image_path)
    pass


### Run Cellpose

In [ ]:
# Additional parameter configs
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
chans = [int(s) for s in channels.value.split(' ')]

# Model
model = CellposeModel(
    gpu=device,
    model_type=base_model.value,
    # pretrained_model=pretrained_model.value
)


In [ ]:
image_path.value

In [ ]:
# Load image & labels
def load_ome_labels(ifile, filename):
    try:
        tif = tifffile.TiffFile(ifile)
        desc = ET.fromstring(tif.pages[0].tags['ImageDescription'].value)
        tree = ET.ElementTree(desc)

        # Check XML tag name for `Channel`
        labels = [elem.get('Name')
                  for elem in tree.iter()
                  if 'Channel' in elem.tag]
        
        ifile.close()
        tif.close()
        return labels

    except (AttributeError, KeyError):
        print('Error retrieving metadata from {}'.format(filename))

    return None

img = tifffile.imread(image_path.value)
ifile = open(image_path.value, 'rb')
channel_labels = load_ome_labels(ifile, image_path.value)
annot_img = {k: v for k, v in zip(channel_labels, img)}

Create RGB image (if applicable):
- R-chan: empty
- G-chan: cyto (`B-catenin`)
- B-chan: nuclei (`DAPI`)

In [ ]:
# Create RGB image:
img_rgb = np.zeros((3, img.shape[-2], img.shape[-1]), dtype=np.int32)
img_rgb[1] = annot_img['B-catenin-AF 488']
img_rgb[2] = annot_img['DAPI']
img_rgb = img_rgb.transpose(1,2,0)

In [ ]:
# Predictions
mask = model.eval(
    img_rgb,
    diameter=diameter.value,
    rescale=rescale.value,
    tile=True,
    channels=chans,
    flow_threshold=flow_threshold.value,
    min_size=min_size.value
)[0]
    

### Save mask predictions

In [ ]:
tifffile.imwrite('mask.tif', mask)

---